In [42]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [43]:
import pandas as pd
import os

path = 'data\\hwu'

df_train = pd.read_csv(os.path.join(path, 'train.csv'), sep=',', header=0, names=['text', 'intent'])
df_val = pd.read_csv(os.path.join(path,'val.csv'), sep=',', header=0, names=['text', 'intent'])
df_test = pd.read_csv(os.path.join(path,'test.csv'), sep=',', header=0, names=['text', 'intent'])

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8954, 2)
Validation shape: (1076, 2)
Test shape: (1076, 2)


,text,intent
0,what alarms do i have set right now,alarm_query
1,checkout today alarm of meeting,alarm_query
2,report alarm settings,alarm_query
3,see see for me the alarms that you have set to...,alarm_query
4,is there an alarm for ten am,alarm_query


In [44]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

intents = pd.concat([df_train['intent'], df_val['intent'], df_test['intent']])

encoder.fit(intents)

df_train['label'] = encoder.transform(df_train['intent'])
df_val['label'] = encoder.transform(df_val['intent'])
df_test['label'] = encoder.transform(df_test['intent'])

df_train

,text,intent,label
0,what alarms do i have set right now,alarm_query,0
1,checkout today alarm of meeting,alarm_query,0
2,report alarm settings,alarm_query,0
3,see see for me the alarms that you have set to...,alarm_query,0
4,is there an alarm for ten am,alarm_query,0
...,...,...,...
8949,how hot is it in miami,weather_query,63
8950,will it snow next week,weather_query,63
8951,am i gonna need rain boots,weather_query,63
8952,should i bring warm clothes,weather_query,63


Nhiệm vụ 1: (Warm-up Ôn bài cũ) Pipeline TF-IDF + Logistic Regression

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score
# 1. Tạo một pipeline với TfidfVectorizer và LogisticRegression
tfidf_lr_pipeline = make_pipeline(
TfidfVectorizer(max_features=5000),
LogisticRegression(max_iter=1000)
)
# 2. Huấn luyện pipeline trên tập train
tfidf_lr_pipeline.fit(df_train['text'], df_train['label'])


# 3. Đánh giá trên tập test
y_pred = tfidf_lr_pipeline.predict(df_test['text'])
y_true = df_test['label']
print(classification_report(y_true, y_pred))

f1_lr = f1_score(y_true, y_pred, average='macro')

              precision    recall  f1-score   support

           0       0.90      0.95      0.92        19
           1       1.00      0.73      0.84        11
           2       0.77      0.89      0.83        19
           3       1.00      0.75      0.86         8
           4       0.92      0.80      0.86        15
           5       0.93      1.00      0.96        13
           6       0.45      0.53      0.49        19
           7       0.89      0.89      0.89        19
           8       0.87      0.68      0.76        19
           9       0.59      0.68      0.63        19
          10       0.67      0.75      0.71         8
          11       0.74      0.89      0.81        19
          12       0.78      0.88      0.82         8
          13       0.83      0.79      0.81        19
          14       0.92      0.63      0.75        19
          15       0.81      0.89      0.85        19
          16       1.00      1.00      1.00        19
          17       1.00    

Nhiệm vụ 2: (Warm-up Ôn bài cũ) Pipeline Word2Vec (Trung bình) + Dense
Layer

In [46]:
from sklearn.metrics import f1_score, classification_report


In [47]:
import numpy as np
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# 1. Huấn luyện mô hình Word2Vec trên dữ liệu text của bạn
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)
# 2. Viết hàm để chuyển mỗi câu thành vector trung bình
def sentence_to_avg_vector(text, model):
    words = text.split()
    vectors = []
    for w in words:
        if w in model.wv:
            vectors.append(model.wv[w])
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    avg_vector = np.mean(vectors, axis=0)
    return avg_vector

# 3. Tạo dữ liệu train/val/test X_train_avg, X_val_avg, X_test_avg
x_train_avg = np.vstack(df_train['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))
x_val_avg = np.vstack(df_val['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))
x_test_avg = np.vstack(df_test['text'].apply(lambda x: sentence_to_avg_vector(x, w2v_model)))


# 4. Xây dựng mô hình Sequential của Keras
num_classes = df_train['label'].nunique()

model = Sequential([
Dense(128, activation='relu', input_shape=(w2v_model.vector_size,)),
Dropout(0.5),
Dense(num_classes, activation='softmax')
])

# 5. Compile, huấn luyện và đánh giá mô hình
#Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

y_train = df_train['label'].values
y_val   = df_val['label'].values
y_test  = df_test['label'].values

#Huấn luyện model
history = model.fit(
    x_train_avg, y_train,
    validation_data=(x_val_avg, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

#Đánh giá model
test_loss_w2v, test_acc = model.evaluate(x_test_avg, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_pred_avg = np.argmax(model.predict(x_test_avg), axis=1)

# Macro F1
f1_avg = f1_score(y_test, y_pred_avg, average='macro')

Epoch 1/20


d:\College\year6s1\nlp-dl\.venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.0226 - loss: 4.1361 - val_accuracy: 0.0288 - val_loss: 4.0980
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0380 - loss: 4.0883 - val_accuracy: 0.0632 - val_loss: 4.0361
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0556 - loss: 4.0117 - val_accuracy: 0.0883 - val_loss: 3.9316
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0654 - loss: 3.9045 - val_accuracy: 0.1041 - val_loss: 3.8021
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0766 - loss: 3.7914 - val_accuracy: 0.1041 - val_loss: 3.6870
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0836 - loss: 3.6964 - val_accuracy: 0.1097 - val_loss: 3.5953
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0966 - loss: 3.6210 - val_accuracy: 0.1134 - val_loss: 3.5429
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1060 - loss: 3.5580 - val_accuracy: 0.1543 - val_

In [48]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.callbacks import EarlyStopping

vocab_size = 20000
max_len = 50
# 1. Tiền xử lý cho mô hình chuỗi
# a. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])
X_train_seq = tokenizer.texts_to_sequences(df_train['text'])
X_val_seq   = tokenizer.texts_to_sequences(df_val['text'])
X_test_seq  = tokenizer.texts_to_sequences(df_test['text'])

# b. Padding: Đảm bảo các chuỗi có cùng độ dài
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post')

# 2. Tạo ma trận trọng số cho Embedding Layer từ Word2Vec
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
# 3. Xây dựng mô hình Sequential với LSTM
lstm_model_pretrained = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix], # Khởi tạo trọng số
        input_length=max_len,
        trainable=False # Đóng băng lớp Embedding
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])
# 4. Compile, huấn luyện (sử dụng EarlyStopping) và đánh giá
lstm_model_pretrained.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history = lstm_model_pretrained.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
test_loss_pretrain, test_acc = lstm_model_pretrained.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test accuracy (Word2Vec + LSTM): {test_acc:.4f}")

y_pred_lstm = np.argmax(
    lstm_model_pretrained.predict(X_test_pad),
    axis=1
)

f1_lstm_pretrained = f1_score(y_test, y_pred_lstm, average='macro')

print("Macro F1 (Word2Vec + LSTM):", f1_lstm_pretrained)


Epoch 1/20


d:\College\year6s1\nlp-dl\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 31ms/step - accuracy: 0.0238 - loss: 4.1197 - val_accuracy: 0.0307 - val_loss: 4.0252
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0350 - loss: 4.0227 - val_accuracy: 0.0446 - val_loss: 3.9395
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0468 - loss: 3.9061 - val_accuracy: 0.0651 - val_loss: 3.7683
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0538 - loss: 3.7995 - val_accuracy: 0.0706 - val_loss: 3.6813
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0590 - loss: 3.7327 - val_accuracy: 0.0781 - val_loss: 3.6196
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0718 - loss: 3.6727 - val_accuracy: 0.0632 - val_loss: 3.6277
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0726 - loss: 3.6434 - val_accuracy: 0.0892 - val_loss: 3.5495
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.0792 - loss: 3.5816 - val_accuracy: 0.09

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM
from tensorflow.keras.callbacks import EarlyStopping

vocab_size = 20000
max_len = 50
# 1. Tiền xử lý cho mô hình chuỗi
# a. Tokenizer: Tạo vocab và chuyển text thành chuỗi chỉ số
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])
X_train_seq = tokenizer.texts_to_sequences(df_train['text'])
X_val_seq   = tokenizer.texts_to_sequences(df_val['text'])
X_test_seq  = tokenizer.texts_to_sequences(df_test['text'])

# b. Padding: Đảm bảo các chuỗi có cùng độ dài
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post')

# 2. Tạo ma trận trọng số cho Embedding Layer từ Word2Vec
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = w2v_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
# 3. Xây dựng mô hình Sequential với LSTM
lstm_model_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=100, # Chọn một chiều embedding, ví dụ 100
        input_length=max_len
# Không có weights, trainable=True (mặc định)
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(num_classes, activation='softmax')
])
# 4. Compile, huấn luyện (sử dụng EarlyStopping) và đánh giá
lstm_model_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history = lstm_model_scratch.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
test_loss_scratch, test_acc = lstm_model_scratch.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test accuracy (Word2Vec + LSTM): {test_acc:.4f}")
y_pred_lstm = np.argmax(
    lstm_model_scratch.predict(X_test_pad),
    axis=1
)

f1_lstm_scratch = f1_score(y_test, y_pred_lstm, average='macro')

print("Macro F1 (Word2Vec + LSTM):", f1_lstm_scratch)




Epoch 1/20


d:\College\year6s1\nlp-dl\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.0168 - loss: 4.1441 - val_accuracy: 0.0177 - val_loss: 4.1296
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0151 - loss: 4.1375 - val_accuracy: 0.0177 - val_loss: 4.1268
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0160 - loss: 4.1346 - val_accuracy: 0.0177 - val_loss: 4.1257
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0152 - loss: 4.1337 - val_accuracy: 0.0177 - val_loss: 4.1247
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.0154 - loss: 4.1333 - val_accuracy: 0.0177 - val_loss: 4.1282
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0157 - loss: 4.1329 - val_accuracy: 0.0177 - val_loss: 4.1256
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.0152 - loss: 4.1330 - val_accuracy: 0.0177 - val_loss: 4.1256
Test accuracy (Word2Vec + LSTM): 0.0177
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Macro F1 (Word2Vec + LSTM):

Nhiệm vụ 5: Đánh giá, So sánh và Phân tích

In [ ]:
print(f1_lr, f1_avg, f1_lstm_pretrained, f1_lstm_scratch)
print(test_loss_w2v, test_loss_pretrain, test_loss_scratch)

0.8352983005857358 0.14141288639720895 0.06178585513830433 0.0005422374429223744
3.1519622802734375 3.381415843963623 4.124667167663574


In [55]:
test_sentences = [
    "can you remind me to not call my mom",
    "is it going to be sunny or rainy tomorrow",
    "find a flight from new york to london but not through paris"
]

true_labels = [
    "reminder_create",
    "weather_query",
    "flight_search"
]

label2intent = {
    i: intent for i, intent in enumerate(encoder.classes_)
}

pred_tfidf = tfidf_lr_pipeline.predict(test_sentences)
pred_tfidf_intents = [label2intent[p] for p in pred_tfidf]

X_test_avg = np.vstack([
    sentence_to_avg_vector(s, w2v_model)
    for s in test_sentences
])

pred_w2v = model.predict(X_test_avg)
pred_w2v_labels = np.argmax(pred_w2v, axis=1)
pred_w2v_intents = [label2intent[p] for p in pred_w2v_labels]

seq_test = tokenizer.texts_to_sequences(test_sentences)
X_test_pad = pad_sequences(seq_test, maxlen=max_len, padding='post')

pred_lstm_pre = lstm_model_pretrained.predict(X_test_pad)
pred_lstm_pre_labels = np.argmax(pred_lstm_pre, axis=1)
pred_lstm_pre_intents = [label2intent[p] for p in pred_lstm_pre_labels]

pred_lstm_scratch = lstm_model_scratch.predict(X_test_pad)
pred_lstm_scratch_labels = np.argmax(pred_lstm_scratch, axis=1)
pred_lstm_scratch_intents = [label2intent[p] for p in pred_lstm_scratch_labels]

print(f"true_labels: {true_labels}\n tfidf: {pred_tfidf_intents}\n w2v: {pred_w2v_intents}\n pretrained-lstm: {pred_lstm_pre_intents}\n scratch_lstm: {pred_lstm_scratch_intents}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
true_labels: ['reminder_create', 'weather_query', 'flight_search']
 tfidf: ['calendar_set', 'weather_query', 'general_negate']
 w2v: ['general_explain', 'calendar_query', 'lists_createoradd']
 pretrained-lstm: ['qa_factoid', 'qa_currency', 'transport_taxi']
 scratch_lstm: ['play_game', 'play_game', 'play_game']
